# Pub/Sub Event Analysis Notebook

Goal: validate event payloads before building a Pub/Sub worker. This helps with event-driven architecture interviews.

In [ ]:
from pathlib import Path
import json
import pandas as pd

events_path = Path('../data/events.jsonl')
events = [json.loads(line) for line in events_path.read_text().splitlines()]
df = pd.DataFrame(events)
df

In [ ]:
def validate_event(event: dict) -> list[str]:
    errors = []
    if not event.get('event_id'):
        errors.append('missing_event_id')
    if not event.get('event_type'):
        errors.append('missing_event_type')
    if event.get('amount', 0) < 0:
        errors.append('negative_amount')
    if event.get('environment') not in {'dev', 'prod'}:
        errors.append('invalid_environment')
    return errors

df['validation_errors'] = [validate_event(event) for event in events]
df['is_valid'] = df['validation_errors'].apply(lambda errors: len(errors) == 0)
df

In [ ]:
valid_events = df[df['is_valid']]
invalid_events = df[~df['is_valid']]

print(f'valid_events={len(valid_events)} invalid_events={len(invalid_events)}')
invalid_events

In [ ]:
event_summary = valid_events.groupby(['event_type', 'environment'], as_index=False).agg(
    event_count=('event_id', 'count'),
    total_amount=('amount', 'sum')
)
event_summary

## Optional: Publish to Pub/Sub

Use this after creating a Pub/Sub topic.

In [ ]:
# from google.cloud import pubsub_v1
# publisher = pubsub_v1.PublisherClient()
# topic_path = publisher.topic_path('YOUR_PROJECT_ID', 'events-topic')
# for event in valid_events.to_dict(orient='records'):
#     publisher.publish(topic_path, json.dumps(event).encode('utf-8'))
# print('published events')